# HR Analytics – Predict Employee Attrition
## Phase 1: Data Understanding

**Author:** Portfolio Project  
**Dataset:** IBM HR Analytics Employee Attrition  
**Objective:** Before any modeling or dashboard work, we need to understand what each column represents, confirm data quality, and identify the prediction target. Skipping this step is one of the most common mistakes in analytics projects — you cannot interpret attrition drivers if you do not know what the fields actually measure.

---

## 1. Load the Dataset

We load the IBM HR dataset from the `data/` folder. Keeping raw data separate from notebooks is a standard project practice — it protects the source file from accidental edits and makes the workflow reproducible.

In [ ]:
# Core libraries for data loading and inspection
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve path relative to project root (works whether notebook is run from notebooks/ or project root)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "WA_Fn-UseC_-HR-Employee-Attrition.csv"

# Load CSV into a DataFrame — each row is one employee record
df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded successfully from: {DATA_PATH}")
print(f"First look at the data:")
df.head()

## 2. Feature Dictionary — What Every Column Means

The IBM HR Analytics dataset contains **35 variables** describing employee demographics, job characteristics, compensation, satisfaction scores, and tenure. Below is a field-by-field explanation based on IBM's published schema and standard HR terminology.

| # | Feature | Type | Description |
|---|---------|------|-------------|
| 1 | **Age** | Numeric | Employee age in years. Useful for workforce planning and age-related retention patterns. |
| 2 | **Attrition** | Categorical | **Target variable.** Whether the employee left the company (`Yes`) or stayed (`No`). |
| 3 | **BusinessTravel** | Categorical | Frequency of work-related travel: `Non-Travel`, `Travel_Rarely`, or `Travel_Frequently`. |
| 4 | **DailyRate** | Numeric | Daily pay rate (masked/scaled value in the dataset). |
| 5 | **Department** | Categorical | Organizational unit: `Sales`, `Research & Development`, or `Human Resources`. |
| 6 | **DistanceFromHome** | Numeric | Miles from home to office. Long commutes often correlate with higher turnover. |
| 7 | **Education** | Ordinal (1–5) | Education level: 1 = Below College, 2 = College, 3 = Bachelor, 4 = Master, 5 = Doctor. |
| 8 | **EducationField** | Categorical | Field of study (e.g., Life Sciences, Medical, Marketing). |
| 9 | **EmployeeCount** | Numeric | Constant field (always 1) — likely a legacy/weight column from IBM's export. |
| 10 | **EmployeeNumber** | Numeric | Unique employee identifier. |
| 11 | **EnvironmentSatisfaction** | Ordinal (1–4) | Satisfaction with physical workspace and environment. Higher = more satisfied. |
| 12 | **Gender** | Categorical | `Male` or `Female`. |
| 13 | **HourlyRate** | Numeric | Hourly pay rate (masked value). |
| 14 | **JobInvolvement** | Ordinal (1–4) | Degree of engagement with the job. 1 = Low, 4 = Very High. |
| 15 | **JobLevel** | Ordinal (1–5) | Hierarchical level within the organization. Higher levels typically mean senior roles. |
| 16 | **JobRole** | Categorical | Specific job title (e.g., Sales Executive, Research Scientist, Manager). |
| 17 | **JobSatisfaction** | Ordinal (1–4) | Satisfaction with the job itself. A direct engagement indicator. |
| 18 | **MaritalStatus** | Categorical | `Single`, `Married`, or `Divorced`. |
| 19 | **MonthlyIncome** | Numeric | Monthly salary in currency units. One of the strongest practical drivers of retention. |
| 20 | **MonthlyRate** | Numeric | Another compensation-related rate (masked). |
| 21 | **NumCompaniesWorked** | Numeric | Number of employers before joining this company. High values may indicate job-hopping tendency. |
| 22 | **Over18** | Categorical | Age confirmation flag (always `Y` in this dataset). |
| 23 | **OverTime** | Categorical | Whether the employee works overtime (`Yes` / `No`). Often linked to burnout and attrition. |
| 24 | **PercentSalaryHike** | Numeric | Most recent percentage salary increase. |
| 25 | **PerformanceRating** | Ordinal | Performance score. In this dataset only ratings 3 (Good) and 4 (Excellent) appear. |
| 26 | **RelationshipSatisfaction** | Ordinal (1–4) | Satisfaction with colleagues and working relationships. |
| 27 | **StandardHours** | Numeric | Standard working hours per period (constant 80 in this dataset). |
| 28 | **StockOptionLevel** | Ordinal (0–3) | Stock option tier. Higher levels reflect stronger long-term retention incentives. |
| 29 | **TotalWorkingYears** | Numeric | Total years of professional experience across all employers. |
| 30 | **TrainingTimesLastYear** | Numeric | Number of training sessions attended in the past year. |
| 31 | **WorkLifeBalance** | Ordinal (1–4) | Perceived balance between work and personal life. |
| 32 | **YearsAtCompany** | Numeric | Total tenure at the current company. |
| 33 | **YearsInCurrentRole** | Numeric | Years in the present job role. Stagnation here can signal career frustration. |
| 34 | **YearsSinceLastPromotion** | Numeric | Years since last promotion. Long gaps often precede resignations. |
| 35 | **YearsWithCurrManager** | Numeric | Years working under the current manager. Manager relationship stability matters for retention. |

### Feature Categories (Analytical Grouping)

For downstream analysis, it helps to mentally group variables:

- **Target:** Attrition  
- **Identifiers / Low-value for modeling:** EmployeeNumber, EmployeeCount, Over18, StandardHours  
- **Demographics:** Age, Gender, MaritalStatus, DistanceFromHome, Education, EducationField  
- **Job & Organization:** Department, JobRole, JobLevel, BusinessTravel, JobInvolvement  
- **Compensation:** MonthlyIncome, DailyRate, HourlyRate, MonthlyRate, PercentSalaryHike, StockOptionLevel  
- **Satisfaction & Engagement:** JobSatisfaction, EnvironmentSatisfaction, RelationshipSatisfaction, WorkLifeBalance  
- **Performance & Development:** PerformanceRating, TrainingTimesLastYear  
- **Tenure & Career Progression:** TotalWorkingYears, YearsAtCompany, YearsInCurrentRole, YearsSinceLastPromotion, YearsWithCurrManager, NumCompaniesWorked  
- **Work Conditions:** OverTime

## 3. Identify the Target Variable

In a supervised classification problem, we must explicitly define what we are trying to predict. Here, the business question is: *"Will this employee leave the organization?"*

**Target variable: `Attrition`**
- `Yes` → Employee left (positive class / attrition event)  
- `No` → Employee retained (negative class)

This is a **binary classification** problem. All other features (except identifiers and constant columns) will serve as predictors after preprocessing.

In [ ]:
TARGET = "Attrition"

print(f"Target variable: {TARGET}")
print(f"Unique values: {df[TARGET].unique().tolist()}")
print("\nClass distribution:")
print(df[TARGET].value_counts())
print("\nClass proportions:")
print(df[TARGET].value_counts(normalize=True).round(4) * 100)

**Early observation:** The dataset is **imbalanced** — roughly 84% of employees stayed while only ~16% left. This is realistic for most organizations (attrition is typically the minority class). We will need to account for this in Phase 5 using metrics like Recall, F1, and ROC-AUC rather than relying on Accuracy alone.

## 4. Dataset Shape

Shape tells us how many employee records and features we are working with. This determines whether we have sufficient sample size for machine learning and whether the feature-to-observation ratio is reasonable.

In [ ]:
n_rows, n_cols = df.shape

print(f"Number of rows (employees): {n_rows:,}")
print(f"Number of columns (features): {n_cols}")
print(f"\nShape: {df.shape}")

With **1,470 employees** and **35 attributes**, we have a manageable dataset size. For tree-based models, this is generally sufficient. The rule of thumb is to have at least 10–20 observations per feature; we comfortably exceed that after removing constant/ID columns.

## 5. Data Types

Understanding dtypes guides preprocessing: categorical variables need encoding, ordinals may be treated as numeric or one-hot depending on model choice, and numeric columns may require scaling for distance-based algorithms like Logistic Regression.

In [ ]:
# Summary of column data types
dtype_summary = df.dtypes.to_frame(name="DataType")
dtype_summary["Non-Null Count"] = df.count()
dtype_summary["Unique Values"] = df.nunique()
dtype_summary

In [ ]:
# Group columns by type for preprocessing planning
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print(f"Numeric columns ({len(numeric_cols)}):")
print(numeric_cols)
print(f"\nCategorical columns ({len(categorical_cols)}):")
print(categorical_cols)

**Preprocessing note:** Several numeric columns are actually **ordinal ratings** (Education, JobSatisfaction, EnvironmentSatisfaction, etc.) on Likert-style scales. We will evaluate in Phase 2 whether to keep them as integers or apply encoding. Constant columns (`EmployeeCount`, `StandardHours`, `Over18`) carry no predictive variance and should be dropped before modeling.

## 6. Duplicate Row Check

Duplicate records would inflate counts and bias attrition rates. We check full-row duplicates and also verify whether `EmployeeNumber` (the unique ID) has any repeats.

In [ ]:
# Full duplicate rows (all columns identical)
duplicate_rows = df.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_rows}")

# Duplicate employee IDs — a more meaningful check for HR data
duplicate_ids = df["EmployeeNumber"].duplicated().sum()
print(f"Duplicate EmployeeNumber values: {duplicate_ids}")

if duplicate_rows == 0 and duplicate_ids == 0:
    print("\n✓ No duplicates found. Each employee appears exactly once.")
else:
    print("\n⚠ Duplicates detected — removal will be handled in Phase 2.")

## 7. Missing Values Check

Missing data can break models and distort EDA. We inspect both total missing count and per-column breakdown. HR systems sometimes have systematic gaps (e.g., promotion dates missing for new hires), so column-level detail matters.

In [ ]:
# Total missing values across the entire dataset
total_missing = df.isnull().sum().sum()
print(f"Total missing values: {total_missing}")

# Per-column missing count and percentage
missing_report = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().sum() / len(df) * 100).round(2)
})
missing_report = missing_report[missing_report["Missing Count"] > 0].sort_values(
    "Missing Count", ascending=False
)

if missing_report.empty:
    print("\n✓ No missing values in any column.")
    display(missing_report)
else:
    print("\nColumns with missing values:")
    display(missing_report)

In [ ]:
# Visual check: any empty strings disguised as missing in object columns?
for col in categorical_cols:
  blank_count = (df[col].astype(str).str.strip() == "").sum()
  if blank_count > 0:
    print(f"{col}: {blank_count} blank/empty strings")

## 8. Data Quality Assessment

A structured quality review before preprocessing saves time later. Below we flag constant columns, near-constant columns, and obvious data integrity issues.

In [ ]:
# Identify constant columns (single unique value — zero information for modeling)
constant_cols = [col for col in df.columns if df[col].nunique() == 1]
print("Constant columns (candidates for removal):")
print(constant_cols)

# Near-constant: PerformanceRating has only 2 levels in this dataset
low_variance_cols = []
for col in numeric_cols:
    if df[col].nunique() <= 2 and col not in constant_cols:
        low_variance_cols.append(col)
print(f"\nLow-cardinality numeric columns: {low_variance_cols}")

# Basic range sanity checks on key numeric fields
print("\nKey numeric ranges:")
key_cols = ["Age", "MonthlyIncome", "YearsAtCompany", "DistanceFromHome", "TotalWorkingYears"]
print(df[key_cols].describe().round(2))

In [ ]:
# pandas profiling-style overview (lightweight alternative)
df.info()

### Data Quality Summary

| Quality Dimension | Finding | Implication |
|-------------------|---------|-------------|
| **Completeness** | No missing values detected | No imputation required; dataset is fully populated |
| **Uniqueness** | No duplicate rows or duplicate employee IDs | No deduplication needed |
| **Validity** | Age (18–60), income (1,009–19,999), tenure values within plausible HR ranges | No obvious data entry errors |
| **Consistency** | Categorical labels are clean (no typos or mixed casing observed) | Straightforward encoding in Phase 2 |
| **Usefulness** | 3 constant columns (`EmployeeCount`, `StandardHours`, `Over18`) | Should be dropped before modeling — they add noise, not signal |
| **Class Balance** | ~16.1% attrition rate (237 Yes / 1,233 No) | Imbalanced target — use appropriate metrics and possibly class weighting |
| **Identifier** | `EmployeeNumber` is unique but not a predictor | Exclude from feature matrix; retain only if needed for tracking |

### Overall Assessment

This is a **clean, analysis-ready dataset** — which is typical for IBM's curated HR analytics sample. The main analytical considerations are not data hygiene but **feature selection** (removing constants/IDs), **class imbalance handling**, and **thoughtful encoding** of categorical and ordinal variables.

From an HR perspective, the dataset covers the right dimensions: compensation, satisfaction, career progression, work-life factors, and demographics. That gives us a solid foundation for attrition driver analysis in Phase 3.

---

## Phase 1 Complete — Key Takeaways

1. **1,470 employees**, **35 features**, binary target `Attrition`  
2. **No missing values** and **no duplicates** — high data quality  
3. **Imbalanced classes** (~16% attrition) — plan evaluation strategy accordingly  
4. **Drop before modeling:** `EmployeeCount`, `StandardHours`, `Over18`, `EmployeeNumber`  
5. **Next step (Phase 2):** Preprocessing — encoding, feature engineering, outlier review, and train-ready dataset preparation

*Awaiting confirmation to proceed to Phase 2 – Data Preprocessing.*